# Finale — Multimodal Product Matching (text + image)

**Goal:** build the best matching system we can by combining what we learned in Parts A, B and C.

```
                ┌── Title ──→ TF-IDF (characters) + sentence embeddings ──┐
Product listing ┤                                                          ├─→ combine ─→ candidates (top-K) ─→ re-rank ─→ match / no match
                └── Image ──→ CLIP embedding ──────────────────────────────┘                                      ↑
                    image_phash, numbers in the title, title length  ("other" signals) ──────────────────────────────┘
```

**The story of this notebook** (each step answers a question):
1. **Baseline:** text similarity + image similarity, averaged 50/50. How good is it?
2. **Experiment 1:** is 50/50 the right weighting? → tune the text/image weight.
3. **Experiment 2:** does adding *semantic* text embeddings to the text branch help?
4. **Experiment 3:** does the cheap `image_phash` column add information?
5. **Experiment 4:** instead of hand-picked weights, can a small model **learn** how to combine all signals? (retrieval + re-ranking)
6. **Ablation:** which components really matter?  7. **Error analysis.**  8. Limitations & future work.

Each experiment follows **Hypothesis → Experiment → Result → Analysis → Conclusion**, and changes **one thing at a time**.

**External resources that influenced this notebook:** CLIP (Radford et al., 2021) and Sentence-BERT (Reimers & Gurevych, 2019) through the
`sentence-transformers` library; scikit-learn (`TfidfVectorizer`, `LogisticRegression`, `HistGradientBoostingClassifier`); the general
"retrieve neighbours with embeddings, then re-rank" recipe that is common in the public Kaggle *Shopee Product Matching* discussions and in retrieval systems in general;
my own Parts B and C. AI-assisted tools were used to write the code skeleton — I can explain every cell.

⚠️ Use a GPU and Internet ON (Kaggle). Embeddings are cached in `results/`, so re-running is fast.

In [ ]:
# Install the extra library if the notebook environment does not have it yet
import importlib, subprocess, sys
for pkg, mod in [("sentence-transformers", "sentence_transformers")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

## 1. Setup — imports, settings and finding the dataset

This cell loads the libraries we need and finds the Shopee dataset automatically
(on Kaggle, Google Colab or your own computer — see `SETUP_GUIDE.md` for how to add the dataset).

In [ ]:
import os, re, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100

SEED = 42                      # a fixed "random seed" so you get the same results every time you run
random.seed(SEED)
np.random.seed(SEED)

# ---------------- Settings you may want to change ----------------
N_GROUPS = 3000                # how many product groups to use (3000 groups ≈ 9,000 listings).
                               # Smaller = faster (use ~500 if you have no GPU). None = whole dataset (slow, needs lots of RAM).
RESULTS_DIR = "results"        # plots and tables are saved here
os.makedirs(RESULTS_DIR, exist_ok=True)


def find_data_dir():
    """Look for the folder that contains train.csv (works on Kaggle, Colab and a local computer)."""
    if os.environ.get("SHOPEE_DIR"):                       # you can force a path with this variable
        return os.environ["SHOPEE_DIR"]
    candidates = [
        "/kaggle/input/shopee-product-matching",           # Kaggle notebook (competition added via 'Add Input')
        "/kaggle/input/competitions/shopee-product-matching",
        "/content/shopee",                                 # Google Colab (see SETUP_GUIDE.md)
        "./shopee-product-matching", "./data", "../data", "../shopee-product-matching",
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    for root, dirs, files in os.walk("/kaggle/input"):     # last resort: search everything under /kaggle/input
        if "train.csv" in files:
            return root
    raise FileNotFoundError("train.csv not found. Follow the steps in SETUP_GUIDE.md to add the dataset.")


DATA_DIR = find_data_dir()
IMG_DIR = os.path.join(DATA_DIR, "train_images")
print("Using data from:", DATA_DIR)

## 2. How we measure success (read this once — it is used in every experiment)

**The task, in one sentence:** for every listing, return the list of listings that are the *same product*.

We compare against the truth (`label_group`: same value = same product). For ONE listing:

| Symbol | Meaning |
|---|---|
| **returned** | how many listings our system says are the same product (it always includes the listing itself) |
| **true** | how many listings really are the same product (also includes itself) |
| **TP** | how many of the *returned* listings are really the same product |

* **Precision** = TP / returned  → "of the matches I claimed, how many were right?"
* **Recall** = TP / true  → "of the real matches, how many did I find?"
* **F1** = 2·TP / (returned + true)  → one number that balances precision and recall

The final score is the **average F1 over all listings**. This is exactly the metric of the original Kaggle competition.

**How a similarity score becomes a decision:** two listings are called the *same product* if `similarity ≥ threshold`.

**Why a "tune" pool and a "test" pool?**
We cannot choose the threshold on the same data we report the final score on — that would be cheating (the score would look
better than it really is). So we split the *product groups* into two pools:
* **tune pool (60 %)** — we try thresholds here and pick the best one,
* **test pool (40 %)** — we only *report* the score here, using the threshold chosen on the tune pool.

We split by *group* (not by listing) so that listings of one product never end up in both pools.

In [ ]:
# Thresholds we will try: 0.10, 0.12, ..., 1.00
THRESHOLDS = np.round(np.arange(0.10, 1.0001, 0.02), 2)


def sample_and_split(df, n_groups=N_GROUPS, test_frac=0.4, seed=SEED):
    """Pick `n_groups` random product groups and split those GROUPS into a 'tune' pool and a 'test' pool."""
    rng = np.random.RandomState(seed)
    groups = np.sort(df["label_group"].unique())
    if n_groups is not None and n_groups < len(groups):
        groups = rng.choice(groups, size=n_groups, replace=False)
    else:
        groups = groups.copy()
    rng.shuffle(groups)
    test_groups = groups[: int(len(groups) * test_frac)]
    sub = df[df["label_group"].isin(groups)].reset_index(drop=True)
    sub["split"] = np.where(sub["label_group"].isin(test_groups), "test", "tune")
    return sub


def sweep_thresholds(sim, labels, thresholds=THRESHOLDS):
    """
    sim    : N x N matrix, sim[i, j] = how similar listing i and listing j are (higher = more similar)
    labels : the true product group of each of the N listings
    For every threshold, return the average precision / recall / F1 over all listings,
    plus how many wrong matches (false positives) and missed matches (false negatives) there were.
    """
    same = labels[:, None] == labels[None, :]          # same[i, j] = True if i and j are truly the same product
    n_true = same.sum(axis=1)                          # number of true matches per listing (including itself)
    rows = []
    for t in thresholds:
        pred = sim >= t                                # our predicted matches
        np.fill_diagonal(pred, True)                   # a listing always matches itself
        n_pred = pred.sum(axis=1)                      # how many we returned per listing
        tp = (pred & same).sum(axis=1)                 # how many of those were correct
        rows.append({
            "threshold": t,
            "precision": float((tp / n_pred).mean()),
            "recall": float((tp / n_true).mean()),
            "f1": float((2 * tp / (n_pred + n_true)).mean()),
            "wrong_matches": int((n_pred - tp).sum()),      # false positives, counted per (listing, candidate)
            "missed_matches": int((n_true - tp).sum()),     # false negatives, counted per (listing, candidate)
        })
    return pd.DataFrame(rows)


def run_experiment(name, representation, similarity, sim_fn, sub, log, extra=None):
    """
    1) compute similarities inside the TUNE pool and pick the threshold with the best F1
    2) compute similarities inside the TEST pool and report the score at that threshold
    `sim_fn(idx)` must return the N x N similarity matrix for the listings `idx`.
    """
    t0 = time.time()
    labels = sub["label_group"].values
    tune_idx = np.where(sub["split"] == "tune")[0]
    test_idx = np.where(sub["split"] == "test")[0]
    sweep_tune = sweep_thresholds(sim_fn(tune_idx), labels[tune_idx])
    sweep_test = sweep_thresholds(sim_fn(test_idx), labels[test_idx])
    best = sweep_tune.loc[sweep_tune["f1"].idxmax()]
    thr = best["threshold"]
    at_thr = sweep_test[sweep_test["threshold"] == thr].iloc[0]
    row = {"Experiment": name, "Representation": representation, "Similarity": similarity,
           "Threshold": thr, "Tune F1": round(best["f1"], 4),
           "Test precision": round(at_thr["precision"], 4), "Test recall": round(at_thr["recall"], 4),
           "Test F1": round(at_thr["f1"], 4), "Seconds": round(time.time() - t0, 1)}
    if extra:
        row.update(extra)
    log.append(row)
    print(f"{name}: threshold={thr:.2f} | tune F1={best['f1']:.4f} | TEST F1={at_thr['f1']:.4f} "
          f"(precision {at_thr['precision']:.3f}, recall {at_thr['recall']:.3f})")
    return {"sweep_tune": sweep_tune, "sweep_test": sweep_test, "threshold": thr, "sim_fn": sim_fn}


def plot_threshold_curves(res, title, save_as=None):
    """Left: precision / recall / F1 versus threshold.  Right: number of wrong and missed matches."""
    st, se, thr = res["sweep_tune"], res["sweep_test"], res["threshold"]
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(st["threshold"], st["precision"], label="precision (tune)", color="tab:blue")
    ax[0].plot(st["threshold"], st["recall"], label="recall (tune)", color="tab:orange")
    ax[0].plot(st["threshold"], st["f1"], label="F1 (tune)", color="tab:green", lw=2.5)
    ax[0].plot(se["threshold"], se["f1"], "--", label="F1 (test)", color="tab:red")
    ax[0].axvline(thr, color="black", ls=":", label=f"chosen threshold = {thr:.2f}")
    ax[0].set_xlabel("threshold"); ax[0].set_ylabel("score"); ax[0].set_title(title); ax[0].legend(loc="center left", fontsize=8)
    ax[1].plot(st["threshold"], st["wrong_matches"] + 1, label="wrong matches (false positives)", color="tab:red")
    ax[1].plot(st["threshold"], st["missed_matches"] + 1, label="missed matches (false negatives)", color="tab:purple")
    ax[1].axvline(thr, color="black", ls=":")
    ax[1].set_yscale("log"); ax[1].set_xlabel("threshold"); ax[1].set_ylabel("count (log scale)")
    ax[1].set_title("Errors vs threshold (tune pool)"); ax[1].legend()
    plt.tight_layout()
    if save_as:
        plt.savefig(os.path.join(RESULTS_DIR, save_as), bbox_inches="tight")
    plt.show()

## 3. Load the data and build the working subset

`train.csv` has ~34,000 listings. Comparing every listing with every other one needs a 34,000 × 34,000 table
(~4.7 GB), so we work with a random **subset of product groups** (`N_GROUPS`). All the real structure is kept
(each group is complete), only the number of "distractor" listings is smaller, so scores will be a little
*higher* than on the full dataset. We say this clearly in the README.

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print("Full training file:", df.shape)

sub = sample_and_split(df, N_GROUPS)
labels_all = sub["label_group"].values
titles = sub["title"].fillna("").astype(str).to_numpy(dtype=object)
tune_idx = np.where(sub["split"] == "tune")[0]
test_idx = np.where(sub["split"] == "test")[0]
print(f"Working subset: {len(sub):,} listings from {sub['label_group'].nunique():,} groups")
print(f"  tune pool: {len(tune_idx):,} listings | test pool: {len(test_idx):,} listings")
sub.head()

## 4. Three pools: train, validation, test

The Finale needs **three** pools because one component is *trained* (the re-ranker) and others are *tuned* (weights, thresholds):

| Pool | Share of groups | Used for |
|---|---|---|
| **train** | 30 % | fitting the learned re-ranker (Experiment 4) |
| **val** | 30 % | choosing weights and thresholds — *every* decision is made here |
| **test** | 40 % | **touched only once, at the end** (ablation table) — the honest final score. It is the *same* test pool as in Parts B and C, so the numbers are comparable |

Splitting is by product group, so a product never appears in two pools.

In [ ]:
rng = np.random.RandomState(SEED + 1)
tune_groups = np.sort(sub.loc[sub["split"] == "tune", "label_group"].unique())
rng.shuffle(tune_groups)
half = len(tune_groups) // 2
sub.loc[sub["label_group"].isin(tune_groups[:half]), "split"] = "train"
sub.loc[sub["label_group"].isin(tune_groups[half:]), "split"] = "val"

pools = {p: np.where(sub["split"] == p)[0] for p in ["train", "val", "test"]}
LAB = {p: labels_all[idx] for p, idx in pools.items()}
for p in pools:
    print(f"{p:5s}: {len(pools[p]):6,} listings, {len(np.unique(LAB[p])):5,} groups")

## 5. Compute the representations (the building blocks)

| Signal | How it is made | Dimension | Lesson from |
|---|---|---|---|
| **TF-IDF words** | word counts weighted by rarity | sparse | Part B (baseline) |
| **TF-IDF characters** (3–5-grams) | robust to typos / spacing | sparse | Part B |
| **MiniLM sentence embedding** | multilingual meaning of the title | 384 | Part B |
| **CLIP image embedding** | what is in the picture | 512 | Part C |
| **phash** | 64-bit fingerprint of the image, already in the CSV | 64 | Part C |

All of these use **no labels** (TF-IDF is fitted only on the title text), so there is no leakage between pools.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from concurrent.futures import ThreadPoolExecutor
from PIL import ImageFile
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

ImageFile.LOAD_TRUNCATED_IMAGES = True
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

# ---- text: TF-IDF ----
tfidf_word = TfidfVectorizer()
X_word = tfidf_word.fit_transform(titles)
tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)
X_char = tfidf_char.fit_transform(titles)
print("TF-IDF words:", X_word.shape, "| TF-IDF characters:", X_char.shape)

# ---- text: sentence embeddings (cached) ----
cache_file = os.path.join(RESULTS_DIR, f"title_emb_minilm_{len(sub)}.npy")
if os.path.exists(cache_file) and np.load(cache_file).shape[0] == len(sub):
    emb_text = np.load(cache_file)
    print("Loaded cached title embeddings", emb_text.shape)
else:
    t0 = time.time()
    st_text = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    emb_text = np.asarray(st_text.encode(list(titles), batch_size=128, show_progress_bar=True, normalize_embeddings=True), dtype=np.float32)
    np.save(cache_file, emb_text)
    print(f"Embedded {len(titles):,} titles in {time.time() - t0:.0f}s")

# ---- image: CLIP embeddings (cached) ----
def load_one(fname):
    img = Image.open(os.path.join(IMG_DIR, fname)).convert("RGB")
    img.thumbnail((512, 512))
    return img


def batches(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


cache_file = os.path.join(RESULTS_DIR, f"img_emb_clip_{len(sub)}.npy")
if os.path.exists(cache_file) and np.load(cache_file).shape[0] == len(sub):
    emb_img = np.load(cache_file)
    print("Loaded cached image embeddings", emb_img.shape)
else:
    t0 = time.time()
    clip_model = SentenceTransformer("clip-ViT-B-32", device=device)
    out = []
    with ThreadPoolExecutor(8) as pool:
        for chunk in tqdm(list(batches(sub["image"].tolist(), 64)), desc="CLIP"):
            imgs = list(pool.map(load_one, chunk))
            out.append(clip_model.encode(imgs, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False))
    emb_img = np.concatenate(out).astype(np.float32)
    np.save(cache_file, emb_img)
    print(f"Embedded {len(sub):,} images in {time.time() - t0:.0f}s")

# ---- image: phash as 64 numbers of +1/-1 scaled to length 1 (cosine = 1 - Hamming/32) ----
def phash_to_vectors(hashes):
    bits = np.array([[int(b) for b in format(int(h, 16), "064b")] for h in hashes], dtype=np.float32)
    return (bits * 2 - 1) / 8.0


emb_phash = phash_to_vectors(sub["image_phash"])

## 6. Similarity matrices

For each pool we build one N × N table per signal: `SIM[pool][signal][i, j]` = how similar listings *i* and *j* are according to that signal.
Everything later (fusion, re-ranking, evaluation) works on these tables.

In [ ]:
def sparse_cosine(X, idx, chunk=1000):
    Xi = X[idx]
    out = np.zeros((len(idx), len(idx)), dtype=np.float32)
    for s in range(0, len(idx), chunk):
        out[s:s + chunk] = (Xi[s:s + chunk] @ Xi.T).toarray()
    return out


SIM = {}
for p, idx in pools.items():
    SIM[p] = {
        "tfidf_word": sparse_cosine(X_word, idx),
        "tfidf_char": sparse_cosine(X_char, idx),
        "minilm": emb_text[idx] @ emb_text[idx].T,
        "clip": emb_img[idx] @ emb_img[idx].T,
        "phash": emb_phash[idx] @ emb_phash[idx].T,
    }
    print(f"{p}: built {len(SIM[p])} similarity tables of size {len(idx):,} x {len(idx):,}")

## 7. Helper functions for tuning and evaluating

* `val_tune(make_S)` — try all thresholds on the **val** pool and return the best one (this is where every decision is made).
* `test_eval(make_S, thr)` — score the **test** pool with a fixed threshold (used only in the final ablation table).

`make_S(pool)` is any function that returns the N × N *final score table* of a pool — a fused similarity or a re-ranker's probabilities.

In [ ]:
def val_tune(make_S):
    sw = sweep_thresholds(make_S("val"), LAB["val"])
    best = sw.loc[sw["f1"].idxmax()]
    return {"f1": float(best["f1"]), "thr": float(best["threshold"]), "precision": float(best["precision"]),
            "recall": float(best["recall"]), "sweep": sw}


def test_eval(make_S, thr):
    return sweep_thresholds(make_S("test"), LAB["test"], [thr]).iloc[0]


def fused(pool, w_text=0.5, a_char=1.0, phash_bonus=0.0):
    """
    Score-level fusion:
      text  = a_char * TF-IDF(characters) + (1 - a_char) * sentence-embedding similarity
      score = w_text * text + (1 - w_text) * CLIP image similarity
      (+ phash_bonus if the two images have an identical phash)
    """
    text = a_char * SIM[pool]["tfidf_char"] + (1 - a_char) * SIM[pool]["minilm"]
    S = w_text * text + (1 - w_text) * SIM[pool]["clip"]
    if phash_bonus > 0:
        S = S + phash_bonus * (SIM[pool]["phash"] >= 0.999)
    return np.clip(S, 0, 1)


val_log = []   # one row per experiment decision (validation results only)

## 8. Baseline

| Item | Choice |
|---|---|
| **Architecture** | two independent branches (text, image) → one combined score |
| **Text representation** | TF-IDF on character 3–5-grams (best classical method from Part B's logic: robust to typos) |
| **Image representation** | CLIP ViT-B/32 embedding (512-d) |
| **Similarity** | cosine similarity inside each branch |
| **Combination** | average of the two similarities: `score = 0.5·text + 0.5·image` |
| **Matching strategy** | `score ≥ threshold` → match; the threshold is the one with the best F1 on the **val** pool |
| **Hyperparameters** | char n-grams (3, 5), min_df = 2, text weight = 0.5, threshold ∈ {0.10 … 1.00} |

(The baseline's *test* score appears in the ablation table at the end; during the experiments we only look at validation scores.)

In [ ]:
base = val_tune(lambda pool: fused(pool, w_text=0.5, a_char=1.0))
print(f"BASELINE (50 % char-TF-IDF + 50 % CLIP): best val threshold = {base['thr']:.2f} | val F1 = {base['f1']:.4f} "
      f"(precision {base['precision']:.3f}, recall {base['recall']:.3f})")
val_log.append({"Experiment": "Baseline 50/50", "setting": "w_text=0.5, char TF-IDF", "val F1": round(base["f1"], 4), "threshold": base["thr"]})

## 9. Experiment 1 — how much should we trust text vs image?

**Hypothesis:** text and image are not equally reliable, so a 50/50 average is not optimal; a better weight (and combining *both* rather than using one) will raise F1.

**Experiment:** change *only* the text weight `w` from 0 (image only) to 1 (text only) in steps of 0.1. Everything else as in the baseline.

In [ ]:
grid = []
for w in np.round(np.arange(0, 1.01, 0.1), 1):
    r = val_tune(lambda pool, w=w: fused(pool, w_text=w, a_char=1.0))
    grid.append({"w_text": w, "val_f1": r["f1"], "threshold": r["thr"]})
grid_df = pd.DataFrame(grid)

plt.figure(figsize=(7, 4))
plt.plot(grid_df["w_text"], grid_df["val_f1"], "o-")
plt.xlabel("weight of the text score (0 = image only, 1 = text only)"); plt.ylabel("validation F1")
plt.title("Experiment 1: text vs image weight"); plt.grid(alpha=0.3)
plt.savefig(os.path.join(RESULTS_DIR, "F_exp1_weights.png"), bbox_inches="tight")
plt.show()

best_w = grid_df.loc[grid_df["val_f1"].idxmax()]
text_only_f1 = grid_df.loc[grid_df["w_text"] == 1.0, "val_f1"].iloc[0]
image_only_f1 = grid_df.loc[grid_df["w_text"] == 0.0, "val_f1"].iloc[0]
print(f"Best text weight = {best_w['w_text']:.1f} → val F1 {best_w['val_f1']:.4f}")
print(f"   baseline 0.5   : {base['f1']:.4f}")
print(f"   text only (1.0): {text_only_f1:.4f}   |   image only (0.0): {image_only_f1:.4f}")
print(f"   gain over the best single modality: {best_w['val_f1'] - max(text_only_f1, image_only_f1):+.4f}")
print(f"   gain over the 50/50 baseline      : {best_w['val_f1'] - base['f1']:+.4f}")
BEST = {"w_text": float(best_w["w_text"]), "a_char": 1.0, "phash_bonus": 0.0}
val_log.append({"Experiment": "Exp 1: tuned text/image weight", "setting": f"w_text={BEST['w_text']:.1f}", "val F1": round(best_w["val_f1"], 4),
                "threshold": best_w["threshold"]})

**Analysis & conclusion — Experiment 1**

*What to look for:* (a) Is the curve flat or does it have a clear peak? (b) Are the two end points (text only, image only) much lower than the best mix? —
if yes, the two modalities are **complementary** (they make different mistakes). (c) Which modality is stronger on its own?

✍️ *Write your conclusion here:* "The best weight was … Fusion improved F1 by … over the best single modality. This supports / contradicts the hypothesis because …"

## 10. Experiment 2 — add semantic sentence embeddings to the text branch

**Hypothesis:** character TF-IDF matches *spelling*, MiniLM embeddings match *meaning* (and other languages). They make different mistakes, so a text branch that mixes both is stronger.

**Experiment:** keep the best weight from Experiment 1 and change *only* the text branch: `text = a·char-TF-IDF + (1−a)·MiniLM`, with `a` ∈ {1, 0.75, 0.5, 0.25, 0}.

In [ ]:
grid = []
for a in [1.0, 0.75, 0.5, 0.25, 0.0]:
    r = val_tune(lambda pool, a=a: fused(pool, w_text=BEST["w_text"], a_char=a))
    grid.append({"a_char": a, "val_f1": r["f1"], "threshold": r["thr"]})
grid_df2 = pd.DataFrame(grid)
display(grid_df2.round(4))

plt.figure(figsize=(6, 3.5))
plt.bar([f"a={a}" for a in grid_df2["a_char"]], grid_df2["val_f1"], color="tab:orange")
plt.ylim(max(0, grid_df2["val_f1"].min() - 0.05), min(1, grid_df2["val_f1"].max() + 0.03))
plt.ylabel("validation F1"); plt.title("Experiment 2: a = share of char-TF-IDF in the text branch")
plt.savefig(os.path.join(RESULTS_DIR, "F_exp2_text_mix.png"), bbox_inches="tight")
plt.show()

best_a = grid_df2.loc[grid_df2["val_f1"].idxmax()]
print(f"Best mix: a = {best_a['a_char']} → val F1 {best_a['val_f1']:.4f} (char TF-IDF only was {grid_df2.loc[grid_df2['a_char'] == 1.0, 'val_f1'].iloc[0]:.4f}, "
      f"MiniLM only {grid_df2.loc[grid_df2['a_char'] == 0.0, 'val_f1'].iloc[0]:.4f})")
BEST["a_char"] = float(best_a["a_char"])
val_log.append({"Experiment": "Exp 2: char TF-IDF + MiniLM text branch", "setting": f"a_char={BEST['a_char']}", "val F1": round(best_a["val_f1"], 4),
                "threshold": best_a["threshold"]})

**Analysis & conclusion — Experiment 2**

*What to look for:* does a mixed text branch beat both pure ones? Is the improvement big or marginal (compare with Part B)? Note that adding the MiniLM
costs extra compute (a neural network over every title) — is the gain worth it?

✍️ *Write your conclusion here.*

### Aside: feature-level fusion vs score-level fusion

We combine **scores** (late fusion). Another way is **feature-level** (early) fusion: glue the two embeddings into one long vector and compare *that*.
If each embedding has length 1 and we scale them by √w and √(1−w), the cosine of the glued vectors is **exactly** `w·cos_text + (1−w)·cos_image`.
So for cosine similarity the two approaches are mathematically the same — check:

In [ ]:
w = BEST["w_text"]
idx = pools["val"][:400]
E_cat = np.hstack([np.sqrt(w) * emb_text[idx], np.sqrt(1 - w) * emb_img[idx]])          # one long vector per listing
S_feature = E_cat @ E_cat.T
S_score = w * (emb_text[idx] @ emb_text[idx].T) + (1 - w) * (emb_img[idx] @ emb_img[idx].T)
print("largest difference between feature-level and score-level fusion:", float(np.abs(S_feature - S_score).max()))

## 11. Experiment 3 — add the "other" information: `image_phash`

**Hypothesis:** two listings with an *identical* perceptual hash have (almost) the same picture, which is strong evidence of the same product. Adding a bonus for
identical hashes should fix some missed matches. (But CLIP already sees identical pictures as nearly identical, so the gain may be small — we test, not assume.)

**Experiment:** add `bonus` to the score when phash cosine ≥ 0.999 (identical hash), for bonus ∈ {0, 0.05, 0.1, 0.2, 0.3, 0.5}. Everything else is fixed.

In [ ]:
grid = []
for b in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    r = val_tune(lambda pool, b=b: fused(pool, w_text=BEST["w_text"], a_char=BEST["a_char"], phash_bonus=b))
    grid.append({"phash_bonus": b, "val_f1": r["f1"], "threshold": r["thr"]})
grid_df3 = pd.DataFrame(grid)
display(grid_df3.round(4))

best_b = grid_df3.loc[grid_df3["val_f1"].idxmax()]
print(f"Best bonus = {best_b['phash_bonus']} → val F1 {best_b['val_f1']:.4f} (without bonus: {grid_df3.iloc[0]['val_f1']:.4f}, "
      f"difference {best_b['val_f1'] - grid_df3.iloc[0]['val_f1']:+.4f})")
n_ident = int((SIM["val"]["phash"] >= 0.999).sum() - len(LAB["val"]))
same_v = LAB["val"][:, None] == LAB["val"][None, :]
n_ident_same = int(((SIM["val"]["phash"] >= 0.999) & same_v).sum() - len(LAB["val"]))
print(f"In the val pool, {n_ident:,} (ordered) pairs have an identical phash; {n_ident_same:,} of them are really the same product "
      f"({100 * n_ident_same / max(n_ident, 1):.0f} % precision of this single rule).")
BEST["phash_bonus"] = float(best_b["phash_bonus"])
val_log.append({"Experiment": "Exp 3: + phash bonus", "setting": f"bonus={BEST['phash_bonus']}", "val F1": round(best_b["val_f1"], 4),
                "threshold": best_b["threshold"]})
print("\nBest hand-tuned fusion so far:", BEST)

**Analysis & conclusion — Experiment 3**

*What to look for:* did F1 move at all? If the bonus is 0 or the gain is tiny, the information in `image_phash` is already contained in the CLIP embedding — a useful negative result.
Check the precision of the "identical phash" rule printed above: if it is far below 100 %, identical hashes do *not* guarantee a match (see Part A).

✍️ *Write your conclusion here.*

## 12. Experiment 4 — let a model *learn* how to combine everything (retrieval + re-ranking)

**Hypothesis:** hand-picked weights treat every pair the same. A small learned model that sees *all* signals for a pair — five similarity scores plus a few cheap extra features —
can learn how much to trust each one and beat the best hand-tuned fusion.

**Design (two stages, as in real search systems):**
1. **Retrieval:** for each listing take the **K = 20 most similar** others according to the best fused score (cheap, no need to look at all N²).
2. **Re-ranking:** a classifier looks at each (listing, candidate) pair and outputs the **probability that they are the same product**. We keep candidates with probability ≥ threshold.

**Features of a pair** (all computed without labels):
| Group | Feature | Meaning |
|---|---|---|
| text | `tfidf_word`, `tfidf_char`, `minilm` | the three title similarities |
| image | `clip` | image similarity |
| other | `phash` | similarity of the image hashes |
| other | `numbers_equal` | 1 if both titles contain exactly the same numbers (sizes, quantities, models) — catches variants |
| other | `len_ratio` | shorter title length / longer title length |

**Training:** only on the **train** pool (labels: same group or not). Threshold chosen on **val**. Two classifiers compared: **logistic regression** (simple, its weights are easy to read) and **gradient boosting** (can model interactions).

In [ ]:
K = 20
title_numbers = [frozenset(re.findall(r"\d+", t.lower())) for t in titles]
title_len = np.array([len(t) for t in titles], dtype=np.float32)

GROUPS = {"text": ["tfidf_word", "tfidf_char", "minilm"], "image": ["clip"], "other": ["phash", "numbers_equal", "len_ratio"]}
ALL_FEATS = sum(GROUPS.values(), [])


def top_candidates(S, k=K):
    """For each listing: the indices of the k most similar OTHER listings (best first)."""
    S2 = S.copy()
    np.fill_diagonal(S2, -np.inf)
    return np.argsort(-S2, axis=1)[:, :k]


PAIRS = {}


def get_pairs(pool):
    """Candidate pairs of a pool with their feature table X and (for training/evaluation) the label y. Cached."""
    if pool in PAIRS:
        return PAIRS[pool]
    idx = pools[pool]
    cand = top_candidates(fused(pool, **BEST))
    n = cand.shape[0]
    rows = np.repeat(np.arange(n), cand.shape[1])
    cols = cand.ravel()
    F = {name: SIM[pool][name][rows, cols] for name in ["tfidf_word", "tfidf_char", "minilm", "clip", "phash"]}
    gi, gj = idx[rows], idx[cols]
    F["numbers_equal"] = np.array([title_numbers[a] == title_numbers[b] for a, b in zip(gi, gj)], dtype=np.float32)
    F["len_ratio"] = np.minimum(title_len[gi], title_len[gj]) / np.maximum(np.maximum(title_len[gi], title_len[gj]), 1)
    X = pd.DataFrame(F)
    y = (LAB[pool][rows] == LAB[pool][cols]).astype(int)
    PAIRS[pool] = (X, y, rows, cols)
    return PAIRS[pool]


def to_matrix(n, rows, cols, p):
    """Put the re-ranker's probabilities back into an N x N table (0 for pairs that were not candidates)."""
    M = np.zeros((n, n), dtype=np.float32)
    M[rows, cols] = p
    np.fill_diagonal(M, 1.0)
    return M


for p in ["train", "val"]:
    X_, y_, _, _ = get_pairs(p)
    same_p = LAB[p][:, None] == LAB[p][None, :]
    n_true = same_p.sum() - len(LAB[p])                    # true (ordered) matches, excluding the listing itself
    print(f"{p}: {len(y_):,} candidate pairs, {100 * y_.mean():.1f} % are true matches | "
          f"candidate recall@{K} = {100 * y_.sum() / n_true:.1f} % (the upper limit of what any re-ranker can find)")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


def make_model(kind):
    if kind == "logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    return HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=SEED)


def rerank_scores(feature_cols, kind="logreg"):
    """Fit on the TRAIN pairs; return make_S(pool) giving the N x N table of match probabilities."""
    X_tr, y_tr, _, _ = get_pairs("train")
    model = make_model(kind).fit(X_tr[feature_cols], y_tr)

    def make_S(pool):
        X, _, rows, cols = get_pairs(pool)
        prob = model.predict_proba(X[feature_cols])[:, 1]
        return to_matrix(len(LAB[pool]), rows, cols, prob)
    return make_S, model


rr = {}
for kind in ["logreg", "gboost"]:
    make_S, model = rerank_scores(ALL_FEATS, kind)
    rr[kind] = {"make_S": make_S, "model": model, "val": val_tune(make_S)}
    print(f"Re-ranker ({kind}): best val threshold {rr[kind]['val']['thr']:.2f} | val F1 {rr[kind]['val']['f1']:.4f} "
          f"(precision {rr[kind]['val']['precision']:.3f}, recall {rr[kind]['val']['recall']:.3f})")

best_hand = val_log[-1]["val F1"]
best_kind = max(rr, key=lambda k_: rr[k_]["val"]["f1"])
print(f"\nBest hand-tuned fusion (val F1): {best_hand:.4f}  |  best re-ranker = {best_kind}: {rr[best_kind]['val']['f1']:.4f}  "
      f"→ difference {rr[best_kind]['val']['f1'] - best_hand:+.4f}")
val_log.append({"Experiment": "Exp 4: learned re-ranker", "setting": f"{best_kind}, K={K}", "val F1": round(rr[best_kind]["val"]["f1"], 4),
                "threshold": rr[best_kind]["val"]["thr"]})

# What did the logistic regression learn to trust? (coefficients on standardised features)
coefs = pd.Series(rr["logreg"]["model"][-1].coef_[0], index=ALL_FEATS).sort_values()
plt.figure(figsize=(6, 3.5))
coefs.plot.barh(color=["tab:red" if c < 0 else "tab:green" for c in coefs])
plt.title("Re-ranker (logistic regression): learned weight of each feature")
plt.xlabel("coefficient (positive = pushes towards 'same product')")
plt.savefig(os.path.join(RESULTS_DIR, "F_exp4_coefficients.png"), bbox_inches="tight")
plt.show()

**Analysis & conclusion — Experiment 4**

*What to look for:* (a) Does the learned model beat the best hand-tuned fusion on validation? By how much? (b) Which features got the largest weights? A big weight on
`numbers_equal` would mean that "do the titles contain the same numbers?" is a strong cue for variants. (c) Candidate recall@K (printed above) limits the
re-ranker: if it is far below 100 %, true matches never reach the second stage — a bigger K or a better first stage would help. (d) Logistic regression vs gradient boosting: is the extra flexibility worth it?

✍️ *Write your conclusion here.*

## 13. Ablation study — which components actually matter?

Now we evaluate **every configuration once on the untouched test pool**, each with the threshold it chose on val.
*Text / Image / Other* tell you which signals the configuration may use ("Other" = phash, numbers in title, title length).
For the learned re-ranker we remove one signal group at a time and **retrain** it.

⚠️ One caveat: the re-ranker's *candidate lists* always come from the best fused score (which uses text and image). So "re-ranker without image" only removes the
image from the *second* stage; the first stage still used it.

In [ ]:
ABL = {}      # results of every configuration
MAKERS = {}   # name -> function that builds its score table (so we can reuse the winner later)


def run_config(name, make_S, text, image, other):
    MAKERS[name] = make_S
    r = val_tune(make_S)
    t = test_eval(make_S, r["thr"])
    ABL[name] = {"Text": text, "Image": image, "Other": other, "thr": r["thr"], "val_f1": r["f1"],
                 "test_precision": float(t["precision"]), "test_recall": float(t["recall"]), "test_f1": float(t["f1"])}
    print(f"{name:55s} val F1 {r['f1']:.4f} → TEST F1 {t['f1']:.4f}")


# --- hand-built fusion configurations ---
run_config("A1  Text only (char TF-IDF)", lambda p: fused(p, w_text=1.0, a_char=1.0), True, False, False)
run_config("A2  Text only (char TF-IDF + MiniLM)", lambda p: fused(p, w_text=1.0, a_char=BEST["a_char"]), True, False, False)
run_config("B   Image only (CLIP)", lambda p: fused(p, w_text=0.0), False, True, False)
run_config("B2  Other only (phash)", lambda p: SIM[p]["phash"], False, False, True)
run_config("C0  Text + Image, 50/50 (BASELINE)", lambda p: fused(p, w_text=0.5, a_char=1.0), True, True, False)
run_config("C1  Text + Image, tuned weight (Exp 1)", lambda p: fused(p, w_text=BEST["w_text"], a_char=1.0), True, True, False)
run_config("C2  + MiniLM in text branch (Exp 2)", lambda p: fused(p, w_text=BEST["w_text"], a_char=BEST["a_char"]), True, True, False)
run_config("D   + phash bonus (Exp 3)", lambda p: fused(p, **BEST), True, True, True)

# --- learned re-ranker and its ablations ---
def rerank_config(name, groups):
    cols = sum([GROUPS[g] for g in groups], [])
    make_S, _ = rerank_scores(cols, best_kind)
    run_config(name, make_S, "text" in groups, "image" in groups, "other" in groups)


rerank_config("E   Re-ranker: text + image + other", ["text", "image", "other"])
rerank_config("E1  Re-ranker without text", ["image", "other"])
rerank_config("E2  Re-ranker without image", ["text", "other"])
rerank_config("E3  Re-ranker without other", ["text", "image"])
rerank_config("E4  Re-ranker: text only", ["text"])
rerank_config("E5  Re-ranker: image only", ["image"])

In [ ]:
abl_df = pd.DataFrame(ABL).T
abl_df["Text"] = abl_df["Text"].map({True: "✓", False: "✗"})
abl_df["Image"] = abl_df["Image"].map({True: "✓", False: "✗"})
abl_df["Other"] = abl_df["Other"].map({True: "✓", False: "✗"})
abl_df = abl_df.rename(columns={"thr": "Threshold (val)", "val_f1": "Val F1", "test_precision": "Test precision",
                                "test_recall": "Test recall", "test_f1": "Test F1"})
abl_df[["Threshold (val)", "Val F1", "Test precision", "Test recall", "Test F1"]] = abl_df[["Threshold (val)", "Val F1", "Test precision", "Test recall", "Test F1"]].astype(float).round(4)
display(abl_df)
abl_df.to_csv(os.path.join(RESULTS_DIR, "finale_ablation.csv"))
pd.DataFrame(val_log).to_csv(os.path.join(RESULTS_DIR, "finale_experiments_val.csv"), index=False)

plt.figure(figsize=(9, 6))
order = abl_df["Test F1"].astype(float)
plt.barh(order.index[::-1], order.values[::-1], color="tab:blue")
for k_, v in enumerate(order.values[::-1]):
    plt.text(v, k_, f" {v:.3f}", va="center", fontsize=8)
plt.xlim(0, 1); plt.xlabel("Test F1"); plt.title("Ablation: test F1 of every configuration")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "F_ablation.png"), bbox_inches="tight")
plt.show()

**How to read the ablation table**
* **A vs B vs C:** compare text only, image only and both → do they complement each other?
* **C0 → C1 → C2 → D:** the contribution of each hand-made improvement (tuned weight, MiniLM, phash).
* **E vs C2/D:** what the learned combination adds.
* **E1, E2, E3:** drop one signal group from the final re-ranker — the **bigger the drop in Test F1, the more that group matters**.
* **B2 (phash only)** shows how far the free column gets you on its own.

✍️ *Write your reading of the table here (cite 2–3 numbers).*

## 14. The final system

We take the configuration with the **best validation F1** (not the best test F1 — the test pool must not be used to choose!).

In [ ]:
final_name = max(ABL, key=lambda k_: ABL[k_]["val_f1"])
print("Final system (best on validation):", final_name)
print(f"   threshold {ABL[final_name]['thr']:.2f} | val F1 {ABL[final_name]['val_f1']:.4f} | "
      f"TEST: precision {ABL[final_name]['test_precision']:.4f}, recall {ABL[final_name]['test_recall']:.4f}, F1 {ABL[final_name]['test_f1']:.4f}")
print(f"Baseline for comparison (C0): TEST F1 {ABL['C0  Text + Image, 50/50 (BASELINE)']['test_f1']:.4f}")

final_make_S = MAKERS[final_name]          # the scoring function of the winning configuration
final_thr = ABL[final_name]["thr"]
S_final = final_make_S("test")

## 15. Error analysis of the final system (test pool)

For every wrong or missed (listing → candidate) pair we ask: *what did each modality say on its own?* Using the thresholds that the **text-only** (A2) and **image-only** (B)
configurations chose on val:

| Text says match? | Image says match? | Meaning for a **wrong match** | Meaning for a **missed match** |
|---|---|---|---|
| ✓ | ✓ | both signals fooled → look-alike variants | both signals said yes but the final system said no |
| ✓ | ✗ | the **title** misled us | image differs, titles agree (different photos of the same product) |
| ✗ | ✓ | the **image** misled us | title differs / noisy, images agree |
| ✗ | ✗ | only the *combination* of weak signals fired | **hardest**: neither title nor image agree |

"Conflicting visual and textual information" = the two middle rows.

In [ ]:
lab_t = LAB["test"]
n_t = len(lab_t)
same_t = lab_t[:, None] == lab_t[None, :]
pred_t = S_final >= final_thr
np.fill_diagonal(pred_t, True)
fp_i, fp_j = np.where(pred_t & ~same_t)       # wrong matches
fn_i, fn_j = np.where(~pred_t & same_t)       # missed matches
print(f"Final system on the test pool: wrong matches = {len(fp_i):,} | missed matches = {len(fn_i):,} (counted as listing → candidate pairs)")

text_S_t = fused("test", w_text=1.0, a_char=BEST["a_char"])
img_S_t = SIM["test"]["clip"]
thr_text = ABL["A2  Text only (char TF-IDF + MiniLM)"]["thr"]
thr_img = ABL["B   Image only (CLIP)"]["thr"]

FP_NAMES = {(True, True): "both text & image say match (look-alike variants)", (True, False): "only the TEXT says match",
            (False, True): "only the IMAGE says match", (False, False): "neither says match (weak signals added up)"}
FN_NAMES = {(True, True): "both say match, final system rejected", (True, False): "only text says match (image differs)",
            (False, True): "only image says match (title differs)", (False, False): "neither says match (hardest case)"}


def bucketize(ii, jj, names):
    t = text_S_t[ii, jj] >= thr_text
    v = img_S_t[ii, jj] >= thr_img
    return np.array([names[(a, b)] for a, b in zip(t, v)])


fp_bucket = bucketize(fp_i, fp_j, FP_NAMES)
fn_bucket = bucketize(fn_i, fn_j, FN_NAMES)
rows = []
for kind, buckets in [("WRONG match", fp_bucket), ("MISSED match", fn_bucket)]:
    for name, cnt in pd.Series(buckets).value_counts().items():
        rows.append({"kind": kind, "what the single modalities said": name, "count": int(cnt),
                     "share of this kind (%)": round(100 * cnt / len(buckets), 1)})
summary = pd.DataFrame(rows)
display(summary)

In [ ]:
def show_pairs_test(ii, jj, notes, heading, pairs_per_row=2):
    nrows = int(np.ceil(len(ii) / pairs_per_row))
    ncols = 2 * pairs_per_row
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.1, nrows * 3.7))
    axes = np.array(axes).reshape(nrows, ncols)
    for ax in axes.ravel():
        ax.axis("off")
    for p, (i, j, note) in enumerate(zip(ii, jj, notes)):
        r, c = divmod(p, pairs_per_row)
        for k, q in enumerate((i, j)):
            row = sub.iloc[pools["test"][q]]
            ax = axes[r, 2 * c + k]
            ax.imshow(load_one(row["image"]))
            ax.set_title(short_title(row["title"], 28, 70), fontsize=7.5)
        axes[r, 2 * c].text(1.0, -0.04, note, transform=axes[r, 2 * c].transAxes, ha="center", va="top", fontsize=7.5, color="navy")
    fig.suptitle(heading, y=1.0)
    plt.tight_layout()
    plt.show()


import textwrap


def short_title(s, width=30, max_chars=80):
    return textwrap.fill(textwrap.shorten(str(s), max_chars, placeholder="…"), width)


def pick_examples(ii, jj, buckets, per_bucket=2, seed=0):
    rng = np.random.RandomState(seed)
    out_i, out_j, notes = [], [], []
    for name in pd.unique(buckets):
        where = np.where(buckets == name)[0]
        for w_ in rng.choice(where, size=min(per_bucket, len(where)), replace=False):
            i, j = ii[w_], jj[w_]
            out_i.append(i); out_j.append(j)
            notes.append(f"{name}\ntext {text_S_t[i, j]:.2f} | image {img_S_t[i, j]:.2f} | final {S_final[i, j]:.2f}")
    return out_i, out_j, notes

### Wrong matches (false positives) — 2 random examples per category

In [ ]:
if len(fp_i):
    ei, ej, en = pick_examples(fp_i, fp_j, fp_bucket)
    show_pairs_test(ei, ej, en, "WRONG matches: the system said 'same product' but they are different products")

### Missed matches (false negatives) — 2 random examples per category

In [ ]:
if len(fn_i):
    ei, ej, en = pick_examples(fn_i, fn_j, fn_bucket)
    show_pairs_test(ei, ej, en, "MISSED matches: same product, but the system said 'different'")

In [ ]:
# Extra evidence for the categories in the task description
def numbers_differ(i, j):
    return title_numbers[pools["test"][i]] != title_numbers[pools["test"][j]]


rng = np.random.RandomState(0)
pick = rng.choice(len(fp_i), size=min(2000, len(fp_i)), replace=False) if len(fp_i) else []
print("VARIANTS: wrong matches whose titles contain different numbers (size / quantity / model): %.0f %%" %
      (100 * np.mean([numbers_differ(fp_i[p], fp_j[p]) for p in pick]) if len(pick) else float("nan")))

short = lambda i: len(titles[pools["test"][i]]) < 25
upper_t = np.triu(np.ones((n_t, n_t), dtype=bool), k=1)
ti, tj = np.where(upper_t & same_t)
pick_all = rng.choice(len(ti), size=min(5000, len(ti)), replace=False)
pick_fn = rng.choice(len(fn_i), size=min(5000, len(fn_i)), replace=False) if len(fn_i) else []
base_rate = np.mean([short(ti[p]) or short(tj[p]) for p in pick_all])
fn_rate = np.mean([short(fn_i[p]) or short(fn_j[p]) for p in pick_fn]) if len(pick_fn) else float("nan")
print("MISSING INFORMATION: a very short title (< 25 characters) is involved in %.0f %% of ALL true pairs but in %.0f %% of the MISSED matches." %
      (100 * base_rate, 100 * fn_rate))

**Failure analysis table** — pick 4–6 real examples from the pictures above and fill the table. (The "typical" row shows the kind of explanation to aim for; replace it with what *you* observed.)

| # | Example (what the pair looks like) | Model predicted | Correct answer | Why did the model fail? | What could fix it? |
|---|---|---|---|---|---|
| typical | two variants of one product (e.g. 250 ml vs 500 ml) with identical stock photo | match | different | text and image are almost identical; only one number differs | give the numbers more weight; extract size/colour attributes and require them to agree |
| 1 | … | … | … | … | … |
| 2 | … | … | … | … | … |
| 3 | … | … | … | … | … |
| 4 | … | … | … | … | … |

**Categories to look for:** similar products · different variants · noisy titles · different images of the same product · visually similar products · missing information · conflicting text/image evidence.
Use the bucket table above for the last one: the share of errors in the "only text" / "only image" rows is how often the modalities **conflicted**.

## 16. Using the final system: "find the matches of this listing"

In [ ]:
def find_matches(posting_id, show=True):
    """Return the listings of the TEST pool that the final system believes are the same product as `posting_id`."""
    pids = sub["posting_id"].values[pools["test"]]
    q = int(np.where(pids == posting_id)[0][0])
    order = np.argsort(-S_final[q])
    matches = [j for j in order if j != q and S_final[q, j] >= final_thr]
    if show:
        rows = [q] + matches[:5]
        fig, axes = plt.subplots(1, len(rows), figsize=(len(rows) * 2.6, 3.4))
        axes = np.atleast_1d(axes)
        for ax, j in zip(axes, rows):
            r = sub.iloc[pools["test"][j]]
            ok = r["label_group"] == sub.iloc[pools["test"][q]]["label_group"]
            ax.imshow(load_one(r["image"]))
            ax.set_title(("QUERY\n" if j == q else f"score {S_final[q, j]:.2f} {'✓' if ok else '✗'}\n") + short_title(r["title"], 22, 44), fontsize=7,
                         color="black" if j == q else ("darkgreen" if ok else "darkred"))
            ax.axis("off")
        plt.tight_layout(); plt.show()
    return sub["posting_id"].values[pools["test"][matches]].tolist()


for q in np.random.RandomState(5).choice(n_t, size=3, replace=False):
    pid = sub["posting_id"].values[pools["test"][q]]
    print(pid, "→ matches:", find_matches(pid))

## 17. Limitations of the final approach

* **Evaluation on a subset** (a few thousand listings instead of all 34 k): fewer "distractors", so scores are optimistic compared with the full dataset. Thresholds tuned on one pool size may be slightly off for another.
* **No model was fine-tuned.** CLIP and MiniLM are used as-is; they were not trained to separate *Shopee product variants*.
* **One global threshold** for all listings, although group sizes range from 2 to ~50.
* **Candidate recall limit:** the re-ranker can only find matches that appear in the top-K candidates.
* **Variants** (size / colour / quantity) remain hard: the embeddings barely react to one changed token or a slightly different colour.
* **Noisy / very short titles and unusual photos** (collages, text overlays) give weak signals for both modalities.
* **Quadratic cost:** the N × N tables work for thousands of listings, not for millions (needs approximate nearest-neighbour search).
* **Single run, single random split:** no confidence intervals, so small differences between configurations (e.g. ±0.005 F1) may be noise.

## 18. Future improvements

* **Fine-tune** the image and text encoders on `label_group` with **metric learning** (ArcFace / contrastive loss) — usually the biggest gain.
* **Hard-negative mining:** train the re-ranker on the *most confusable* different-product pairs (variants).
* **Better encoders:** larger CLIP (ViT-L/14), multilingual CLIP for titles and images in a shared space, an Indonesian BERT.
* **Attribute extraction:** parse size / colour / quantity / brand from titles and require them to agree.
* **Graph / clustering post-processing:** use the transitivity of "same product" (connected components or neighbours-of-neighbours / query expansion).
* **Per-listing thresholds** (e.g. relative to the best neighbour's score).
* **Efficiency:** FAISS / HNSW approximate search, mixed-precision inference, caching embeddings.
* **Evaluate on the full dataset** with confidence intervals (several random splits).

## 19. Final report (fill in and export as `report.pdf`)

**1. Approach** — Two-stage retrieve-and-rerank system. Stage 1: title similarity (char TF-IDF, optionally mixed with multilingual MiniLM embeddings) and image similarity (CLIP ViT-B/32) are fused by a weighted average to pick the K = 20 best candidates per listing.
Stage 2: a small classifier looks at the pair's similarities plus `image_phash`, "same numbers in title" and title-length ratio, and outputs the probability of a match; probability ≥ threshold ⇒ match.

**2. Experiments** — *(copy the validation table `finale_experiments_val.csv` and one sentence per experiment: hypothesis → result → conclusion)*

**3. Analysis** — *(why fusion helped or not, why MiniLM / phash / re-ranker helped or not — cite numbers)*

**4. Error analysis** — *(the failure-analysis table and the bucket table of section 15)*

**5. Ablation study** — *(paste the ablation table `finale_ablation.csv` and your reading of it)*

**6. Limitations** — see section 17.

**7. Future improvements** — see section 18.

**Resources used:** CLIP, Sentence-BERT / `sentence-transformers`, scikit-learn, public Kaggle Shopee competition discussions for the general retrieve-and-rerank idea, AI-assisted coding tools for the initial code (all code understood and explainable).

## 20. Quick-reference: what to say in the interview

* **Why cosine similarity of normalised vectors?** It measures direction (content), not size; for normalised vectors it equals a dot product, so thousands of comparisons are one matrix multiplication.
* **Why a tune / val / test split by group?** Same product in two pools would leak the answer; the threshold and weights must be chosen without looking at the test data.
* **Why F1 and not accuracy?** More than 99.9 % of pairs are non-matches; "always no" would have 99.9 % accuracy.
* **Why two stages?** Comparing every pair with a learned model is too slow; cheap similarity picks K candidates, the learned model refines only those.
* **Why does fusion help?** Titles and images fail in different cases (different photos of one product vs look-alike variants), so one can correct the other.
* **What is the weakest part?** Variants — and the fact that nothing is fine-tuned on the task.